# vid-bid on a free GPU

Replace an object in a video with your product: for example, swap the can in an ad for a Pepsi Black can. This notebook runs the open-source pipeline on Google Colab's free T4 GPU (it also works on Kaggle). No accounts, API keys or per-use costs.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. The **realistic** engine (Wan 2.1 VACE, open weights) re-renders the object with the scene's lighting and motion. It downloads about 20 GB of weights on first use and takes several minutes per ~3 s part on a T4.
3. The **local** engine is the fast option: minutes for a 30 s clip, and the label is pixel-exact.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime → Change runtime type → T4 GPU"

In [ ]:
#@title Get vid-bid
BRANCH = "main"  #@param {type:"string"}
!git clone -q --depth 1 -b {BRANCH} https://github.com/mayanksagar26/vid-bid /content/vid-bid
%cd /content/vid-bid
!pip install -q -r requirements.txt

In [ ]:
#@title Inputs
VIDEO_URL = "https://youtu.be/8ivQOS4r-K8"  #@param {type:"string"}
OBJECT = "can"  #@param {type:"string"}
ENGINE = "vace"  #@param ["vace", "local"]
DESCRIBE = "a black Pepsi Black can"  #@param {type:"string"}
#@markdown Leave VIDEO_URL empty to upload a video file instead (YouTube sometimes blocks cloud downloads).
#@markdown You'll be asked to upload the product photo (PNG with transparency, or a plain background).
import os
try:
    from google.colab import files
    up = lambda: list(files.upload().keys())[0]
except ImportError:  # Kaggle / Jupyter: put files next to the notebook and set the names here
    up = None
VIDEO = None
if not VIDEO_URL:
    print("Upload the video:")
    VIDEO = up() if up else "video.mp4"
print("Upload the product photo:")
PRODUCT = up() if up else "product.png"

In [ ]:
#@title Replace
src = f'--url "{VIDEO_URL}"' if VIDEO_URL else f'--video "{VIDEO}"'
!python -m app.cli {src} --product "{PRODUCT}" --object "{OBJECT}" --engine {ENGINE} --describe "{DESCRIBE}" --out /content/result.mp4

In [ ]:
#@title Watch and download
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open("/content/result.mp4", "rb").read()).decode()
display(HTML(f'<video width="720" controls src="data:video/mp4;base64,{data}"></video>'))
try:
    from google.colab import files
    files.download("/content/result.mp4")
except ImportError:
    pass